In [2]:
import os
from pathlib import Path

import gspread
import pandas as pd
from google.oauth2.service_account import Credentials

# From URL: https://docs.google.com/spreadsheets/d/<SPREADSHEET_ID>/edit
SPREADSHEET_ID = "140vMgvckcXRfl13sVGedGWV740JZw-R9QlHbfpJXczE"
SHEET_NAME = "1. Grundbetreuung Updated"

# credentials_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
credentials_path = "./config/gsheet-creds.json"
# Or set explicitly: credentials_path = Path.home() / "secrets" / "service-account.json"
if not credentials_path:
    raise FileNotFoundError(
        "Set GOOGLE_APPLICATION_CREDENTIALS to your service account JSON path"
    )
credentials_path = Path(credentials_path).expanduser()

import duckdb

# Run the first gspread config cell so `credentials_path`, `SPREADSHEET_ID`, and `SHEET_NAME` exist.


def _sql_literal(s: str) -> str:
    return s.replace("'", "''")


key_path = _sql_literal(str(credentials_path.resolve()))

duck = duckdb.connect()
duck.execute("INSTALL gsheets FROM community;")
duck.execute("LOAD gsheets;")
duck.execute(
    f"""
CREATE OR REPLACE SECRET gsheet_sa (
    TYPE gsheet,
    PROVIDER key_file,
    FILEPATH '{key_path}'
);
"""
)

In [13]:
duck.sql(
    f"""
    create or replace table padoa_prices as 
    SELECT * FROM read_gsheet(
    '{_sql_literal(SPREADSHEET_ID)}',
    sheet='{_sql_literal(SHEET_NAME)}',
    all_varchar=true, range='A3:N'
)
"""
)

In [7]:
wochenliste_id = '1Rz_dGw4y1b1ym1QN-XFV9W5FtfZShmVQKeq0bRQgnQM'
w_sheet_name = 'All clients'

In [ ]:

duck.sql(
    f"""
    create or replace table basic_care_prices as
    SELECT *
    FROM read_gsheet(
    '{_sql_literal(wochenliste_id)}',
    sheet='{_sql_literal(w_sheet_name)}',
    all_varchar=true
    )
"""
)

In [26]:
duck.sql(
    """
    select  * from basic_care_prices  order by id_easybill
    """
)

┌─────────────┬────────────────────────────────────────────────────────────────────┬───────────┬────────────┬──────────────┬───────────────┬─────────────────┬────────────────────────────────┬──────────────────────────────┬──────────────────────────────┬────────────────────────────────────────────┬───────────────────────┬────────────────────────────────────────────────────────┬────────────────┬─────────────────────┬──────────────────────┬────────────────┬─────────────┬───────────────┐
│ id_easybill │                             Firmenname                             │ ID Nummer │  Standort  │ Dokument: ID │ Dokument: Typ │ Dokument: Datum │ Dokument: Leistungsdatum Datum │ Dokument: Leistungsdatum von │ Dokument: Leistungsdatum bis │ Dokument: Leistungsdatum Benutzerdefiniert │ Posten: Artikelnummer │              Posten: Artikelbeschreibung               │  Posten: Typ   │ Posten: Nettobetrag │ Posten: Bruttobetrag │ Posten: Anzahl │ is_pauschal │ hourly prices │
│   varchar   │       

In [21]:
duck.sql(
    """
    select distinct on (id_easybill) * from basic_care_prices  order by id_easybill
    """
)

┌─────────────┬────────────────────────────────────────────────────────────────────┬────────────┬───────────────────┬──────────────┬───────────────┬─────────────────┬────────────────────────────────┬──────────────────────────────┬──────────────────────────────┬────────────────────────────────────────────┬───────────────────────┬──────────────────────────────────────────────────┬────────────────┬─────────────────────┬──────────────────────┬────────────────┬─────────────┬───────────────┐
│ id_easybill │                             Firmenname                             │ ID Nummer  │     Standort      │ Dokument: ID │ Dokument: Typ │ Dokument: Datum │ Dokument: Leistungsdatum Datum │ Dokument: Leistungsdatum von │ Dokument: Leistungsdatum bis │ Dokument: Leistungsdatum Benutzerdefiniert │ Posten: Artikelnummer │           Posten: Artikelbeschreibung            │  Posten: Typ   │ Posten: Nettobetrag │ Posten: Bruttobetrag │ Posten: Anzahl │ is_pauschal │ hourly prices │
│   varchar   │   

In [14]:
duck.sql(
    """
    create or replace table full_basic_care as 
select * from read_csv('/Users/adrienblanquer/code/bas-utils/basic_care/full_basic_care.csv')
    """)

In [ ]:
duck.sql(
    "select * from full_basic_care"
)

┌──────────────────┬─────────────────┬────────────────────────────────────────────────────────────────────┬───────────┬───────────────────────────────────────────────┬───────────┬─────────┬──────────┬─────────┬───────────────┬──────────────────┬─────────────┬─────────┬────────────────┬────────────────────────┬─────────────────┬────────────┬───────────┬────────────┐
│ mother_client_id │ child_client_id │                             firm_name                              │ Standort  │                   Anschrift                   │   arzt    │  fasi   │ typ_amed │ typ_asi │ betreuungsart │ direkte_leistung │ mitarbeiter │   bg    │ vertragsbeginn │ start_aktuelle_periode │ vertragsstunden │ davon_amed │ davon_asi │ davon_apsy │
│      int64       │      int64      │                              varchar                               │  varchar  │                    varchar                    │  varchar  │ varchar │ varchar  │ varchar │    varchar    │     varchar      │   varchar   │ varc

In [39]:
duck.sql(
    """
    select fbc.bg, fbc.total_mitarbeiter, bcp.* 
    from basic_care_prices bcp
    join (
        select mother_client_id, bg, mitarbeiter, child_client_id,
               sum(replace(mitarbeiter, ',' ,'.')::float) over(partition by child_client_id) as total_mitarbeiter
        from (select distinct on(mother_client_id) mother_client_id, bg, mitarbeiter, child_client_id from full_basic_care)
    ) fbc
        on mother_client_id = id_easybill
    order by id_easybill
    """
)

┌─────────┬───────────────────┬─────────────┬────────────────────────────────────────────────────────────────────┬───────────┬──────────┬──────────────┬───────────────┬─────────────────┬────────────────────────────────┬──────────────────────────────┬──────────────────────────────┬────────────────────────────────────────────┬───────────────────────┬─────────────────────────────────────────────┬────────────────┬─────────────────────┬──────────────────────┬────────────────┬─────────────┬───────────────┐
│   bg    │ total_mitarbeiter │ id_easybill │                             Firmenname                             │ ID Nummer │ Standort │ Dokument: ID │ Dokument: Typ │ Dokument: Datum │ Dokument: Leistungsdatum Datum │ Dokument: Leistungsdatum von │ Dokument: Leistungsdatum bis │ Dokument: Leistungsdatum Benutzerdefiniert │ Posten: Artikelnummer │         Posten: Artikelbeschreibung         │  Posten: Typ   │ Posten: Nettobetrag │ Posten: Bruttobetrag │ Posten: Anzahl │ is_pauschal │ hour

In [43]:
# % adjustment needed to bring each client in line with the padoa reference price.
# Match criteria: bg (WZ Group) x total_employees band x price model (pauschal/variabel).
#   pauschal -> the "Wenn pauschal:" price is for the WHOLE package, so compare the client's combined
#               AM-R + AS-R Nettobetrag (summed per invoice / id_easybill) against the combined
#               "ArbMed und ArbSich" reference row. -> one row per invoice (product = 'Paket').
#   variabel -> per posten: AM-R vs Stundensatz ArbMed, AS-R vs Stundensatz ArbSich. When the employee
#               band has no variabel rate in padoa (it prescribes pauschal there), fall back to the
#               default hourly rates: ArbMed 120, ArbSich 80. -> one row per posten.
# adjustment_pct = padoa_ref / current - 1  (delta %; positive => price must rise to match padoa).
duck.sql(
    r"""
    with padoa_var as (   -- per-product variabel hourly rates
        select trim("WZ Group") as bg, trim("Angebot") as angebot,
            regexp_extract("Mitarbeiterzahl", '(\d+)', 1)::int                          as emp_low,
            case when "Mitarbeiterzahl" ilike '%und mehr%' then 1000000
                 else regexp_extract("Mitarbeiterzahl", '-\s*(\d+)', 1)::int end        as emp_high,
            try_cast(replace("Stundensatz ArbMed",  ',', '.') as double)                as rate_arbmed,
            try_cast(replace("Stundensatz ArbSich", ',', '.') as double)                as rate_arbsich
        from padoa_prices
        where trim("Preismodel") = 'Variabel' and trim("Angebot") in ('ArbMed', 'ArbSich')
    ),
    padoa_pkg as (        -- combined "ArbMed und ArbSich" pauschal package price
        select trim("WZ Group") as bg,
            regexp_extract("Mitarbeiterzahl", '(\d+)', 1)::int                          as emp_low,
            case when "Mitarbeiterzahl" ilike '%und mehr%' then 1000000
                 else regexp_extract("Mitarbeiterzahl", '-\s*(\d+)', 1)::int end        as emp_high,
            try_cast(replace(regexp_replace("Wenn pauschal: ", '[^0-9,]', '', 'g'), ',', '.') as double) as pauschal_price
        from padoa_prices
        where trim("Angebot") = 'ArbMed und ArbSich' and trim("Preismodel") = 'Pauschal'
    ),
    fbc as (
        select mother_client_id, bg, child_client_id,
               sum(replace(mitarbeiter, ',' ,'.')::float) over(partition by child_client_id) as total_mitarbeiter
        from (select distinct on(mother_client_id) mother_client_id, bg, mitarbeiter, child_client_id from full_basic_care)
    ),
    bc as (
        select fbc.bg as bg, fbc.total_mitarbeiter, bcp.id_easybill, bcp."Firmenname" as firmenname,
            case when bcp."Posten: Artikelnummer" like 'AM%' then 'ArbMed'
                 when bcp."Posten: Artikelnummer" like 'AS%' then 'ArbSich' end          as angebot,
            (lower(bcp.is_pauschal) = 'true')                                            as pauschal,
            try_cast(replace(regexp_replace(bcp."Posten: Nettobetrag", '[^0-9,]', '', 'g'), ',', '.') as double) as netto,
            try_cast(replace(bcp."hourly prices", ',', '.') as double)                   as hourly
        from basic_care_prices bcp
        join fbc on fbc.mother_client_id = bcp.id_easybill
    ),
    -- pauschal: sum the postens per invoice, match to the combined package price
    pauschal as (
        select bc.bg, any_value(bc.total_mitarbeiter) as employees, bc.id_easybill,
               any_value(bc.firmenname) as firmenname, 'pauschal' as model, 'Paket' as product,
               sum(bc.netto) as current_price
        from bc where bc.pauschal group by bc.bg, bc.id_easybill
    ),
    pauschal_m as (
        select ps.bg, ps.employees, ps.id_easybill, ps.firmenname, ps.model, ps.product,
               ps.current_price, pkg.pauschal_price as padoa_ref
        from pauschal ps
        left join padoa_pkg pkg
               on pkg.bg = ps.bg
              and ps.employees >= pkg.emp_low and ps.employees <= pkg.emp_high
    ),
    -- variabel: per posten, per-product hourly rate (defaults 120 / 80 when no variabel band applies)
    variabel as (
        select bc.bg, bc.total_mitarbeiter as employees, bc.id_easybill, bc.firmenname,
               'variabel' as model, bc.angebot as product, bc.hourly as current_price,
               case when bc.angebot = 'ArbMed' then coalesce(pv.rate_arbmed, 120)
                    else coalesce(pv.rate_arbsich, 80) end as padoa_ref
        from bc
        left join padoa_var pv
               on pv.bg = bc.bg and pv.angebot = bc.angebot
              and bc.total_mitarbeiter >= pv.emp_low and bc.total_mitarbeiter <= pv.emp_high
        where not bc.pauschal
    )
    select bg, employees, id_easybill, firmenname, model, product, current_price, padoa_ref,
           round((padoa_ref / nullif(current_price, 0) - 1) * 100, 1) as adjustment_pct
    from (select * from pauschal_m union all select * from variabel)
    order by id_easybill, product
    """
).to_csv('basic_care_discounts.csv')